# Naive Bayes Regression using Sentiment Analysis Dataset

### Download Dataset from here
### https://www.kaggle.com/code/lakshmi25npathi/sentiment-analysis-of-imdb-movie-reviews/input

In [21]:
import pandas as pd
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [22]:
# One review
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

## Text Cleaning

1. sample 10,000 rows
2. Remove html tags  (ex:- <html, <p, br> etc) 
3. Remove special characters   (ex:- $, #, %, ^, &, * etc..) 
4. Converting every thing to Lower-Case   (ex:- 'Hello This' becomes 'hello this')
5. Removing Stop words  ( and, i, am, are, is etc...)
6. Apply Stemming  ( means words ex-:  liking, liked, likes all become like)

In [23]:
# instead of 50,000 rows take a sample of 10,000 rows and work on it
df = df.sample(10000)

In [24]:
df.shape

(10000, 2)

In [25]:
# check missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10000 entries, 9059 to 1332
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     10000 non-null  object
 1   sentiment  10000 non-null  object
dtypes: object(2)
memory usage: 234.4+ KB


In [26]:
df['sentiment'] = df['sentiment'].replace({'positive': 1, 'negative': 0})
# df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

/tmp/ipykernel_13712/559549869.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'positive': 1, 'negative': 0})


In [27]:
df.head()

,review,sentiment
9059,"Full marks for the content of this film, as a ...",0
28271,My mom brought me this movie on a DVD. A guy i...,0
18229,Legendary hammy and arrogant horror movie star...,1
49488,"This is a kind of genre thing, meaning you eit...",0
46194,I am decidedly not in the target audience for ...,1


In [28]:
# Apply Regular Expression library to remove html tags from a single row
import re
clean = re.compile('<.*?>')
re.sub(clean, '', df.iloc[2].review) # Replace review text from the 3rd row (0-indexed) with empty string ''

'Legendary hammy and arrogant horror movie star Conrad Radzoff (splendidly played with wicked sardonic aplomb by Ferdy Mayne) dies of a heart attack. A bunch of drama school students steal Radzoff\'s corpse from its crypt and take it to a rundown mansion so they can party with it. Radzoff comes back to life and picks off the rude youths for desecrating his grave. Writer/director Norman Thaddeus Vane concocts a fresh, original, and even pretty stylish spin on the usual body count premise, offers a neat evocation of the glitzy Hollywood milieu, and does a sturdy job of maintaining a pleasingly misty and spooky ooga-booga atmosphere. The kill set pieces deliver the grisly goods, with a gal being set on fire, a juicy decapitation (the severed head rolls right down the stairs and onto the lawn so a raven can peck away at it!), and another poor lass being crushed with a levitating coffin rating as the definite gruesome highlights. Kudos are also in order for the stellar cast of familiar B-fl

In [29]:
# Function to clean html tags from all rows 
def clean_html(text):
    clean = re.compile('<.*?>')
    return re.sub(clean, '', text)

In [30]:
df['review'] = df['review'].apply(clean_html)

In [31]:
# Converting Every word to Lower-case
def convert_lower(text):
    return text.lower()

In [32]:
df['review'] = df['review'].apply(convert_lower)

In [33]:
# Function to remove Special Characters
def remove_special(text):
    x='' #Empty String 

    for i in text:
        if i.isalnum():
            x=x+i
        else:
            x=x+' '
    return x

In [34]:
# remove from given text
remove_special('@ # hello this is moin  $ how are you brother * hey perhaps you do not know about - this matter ')

'    hello this is moin    how are you brother   hey perhaps you do not know about   this matter '

In [35]:
# remove from all rows 
df['review'] = df['review'].apply(remove_special)

In [36]:
# Remove the Stop words
import nltk  # nltk means natural language toolkit
from nltk.corpus import stopwords

In [37]:
stopwords.words('english') # list of english language stopwords that need to be removed

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [38]:
df

,review,sentiment
9059,full marks for the content of this film as a ...,0
28271,my mom brought me this movie on a dvd a guy i...,0
18229,legendary hammy and arrogant horror movie star...,1
49488,this is a kind of genre thing meaning you eit...,0
46194,i am decidedly not in the target audience for ...,1
...,...,...
29499,dolemite is one of the best movies featuring a...,1
34028,nynke is a classy filmed movie in the same sty...,1
13156,a great and truly independent film that hit mo...,1
33991,five years on from the tenko survivors returni...,1


In [39]:
def remove_stopwords(text):
    x = []
    for i in text.split():
        if i not in stopwords.words('english'):
            x.append(i)
    y=x[:]
    x.clear()
    return y

In [40]:
df['review'] = df['review'].apply(remove_stopwords)

In [41]:
df

,review,sentiment
9059,"[full, marks, content, film, brit, aware, segr...",0
28271,"[mom, brought, movie, dvd, guy, rental, recomm...",0
18229,"[legendary, hammy, arrogant, horror, movie, st...",1
49488,"[kind, genre, thing, meaning, either, like, 19...",0
46194,"[decidedly, target, audience, film, man, nearl...",1
...,...,...
29499,"[dolemite, one, best, movies, featuring, pimp,...",1
34028,"[nynke, classy, filmed, movie, style, oscar, w...",1
13156,"[great, truly, independent, film, hit, emotion...",1
33991,"[five, years, tenko, survivors, returning, hom...",1


In [42]:
# Perform Stemming
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

y = []
def stem_words(text):
    for i in text:
        y.append(ps.stem(i))
    z=y[:]
    y.clear()
    return z

In [43]:
stem_words(['loved', 'loving', 'loves', 'lowered', 'lowering', 'it', 'its'])

['love', 'love', 'love', 'lower', 'lower', 'it', 'it']

In [44]:
df['review'] = df['review'].apply(stem_words)

In [45]:
df

,review,sentiment
9059,"[full, mark, content, film, brit, awar, segreg...",0
28271,"[mom, brought, movi, dvd, guy, rental, recomme...",0
18229,"[legendari, hammi, arrog, horror, movi, star, ...",1
49488,"[kind, genr, thing, mean, either, like, 1950, ...",0
46194,"[decidedli, target, audienc, film, man, nearli...",1
...,...,...
29499,"[dolemit, one, best, movi, featur, pimp, hero,...",1
34028,"[nynk, classi, film, movi, style, oscar, win, ...",1
13156,"[great, truli, independ, film, hit, emot, carr...",1
33991,"[five, year, tenko, survivor, return, home, ma...",1


In [46]:
# Join all list using join_back function
def join_back(list_input):
    return " ".join(list_input)

In [47]:
df['review'] = df['review'].apply(join_back)

In [48]:
df['review']

9059     full mark content film brit awar segreg us nav...
28271    mom brought movi dvd guy rental recommend fact...
18229    legendari hammi arrog horror movi star conrad ...
49488    kind genr thing mean either like 1950 music lo...
46194    decidedli target audienc film man nearli 50 re...
                               ...                        
29499    dolemit one best movi featur pimp hero take ma...
34028    nynk classi film movi style oscar win film cha...
13156    great truli independ film hit emot carri anoth...
33991    five year tenko survivor return home marion do...
1332     although fan heather thoma old bikini poster a...
Name: review, Length: 10000, dtype: object

In [49]:
X = df.iloc[:,0:1].values

In [50]:
X.shape

(10000, 1)

In [51]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=1000)

X = cv.fit_transform(df['review']).toarray()
X.shape  

(10000, 1000)

In [52]:
print('Every row unique words count : ', X[1])
print('maximum a word came in a row : ',X[1].max())
print('mean value of a row : ',X[1].mean())


Every row unique words count :  [0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 2 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 2 0
 0 1 0 0 2 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0
 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 4 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

In [53]:
y = df.iloc[:,-1].values
y.shape

(10000,)

In [54]:
y  #Positive_Review=1, Negative-Review=0

array([0, 0, 1, ..., 1, 1, 0])

In [55]:
# from X, y split into Training_Set && Test_Set(Already know the result)

In [56]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [57]:
print('X_train shape : ', X_train.shape)
print('X_test shape : ', X_test.shape)
print('y_train shape : ', y_train.shape)
print('y_test shape : ', y_test.shape)

X_train shape :  (8000, 1000)
X_test shape :  (2000, 1000)
y_train shape :  (8000,)
y_test shape :  (2000,)


In [58]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
clf1 = GaussianNB()
clf2 = MultinomialNB()
clf3 = BernoulliNB()

In [59]:
clf1.fit(X_train,y_train)
clf2.fit(X_train,y_train)
clf3.fit(X_train,y_train)

,alpha,1.0
,force_alpha,True
,binarize,0.0
,fit_prior,True
,class_prior,None


In [60]:
y_pred1 = clf1.predict(X_test)
y_pred2 = clf2.predict(X_test)
y_pred3 = clf3.predict(X_test)

In [61]:
print('y_test Shape : ', y_test.shape)
print('y_pred1 Shape : ', y_pred1.shape)
print('y_pred2 Shape : ', y_pred2.shape)
print('y_pred3 Shape : ', y_pred3.shape)

y_test Shape :  (2000,)
y_pred1 Shape :  (2000,)
y_pred2 Shape :  (2000,)
y_pred3 Shape :  (2000,)


In [63]:
from sklearn.metrics import accuracy_score
print('GaussianNB    : ',  accuracy_score(y_test, y_pred1))
print('MultinomialNB : ',  accuracy_score(y_test, y_pred2))
print('BernoulliNB   : ',  accuracy_score(y_test, y_pred3))

GaussianNB    :  0.7705
MultinomialNB :  0.832
BernoulliNB   :  0.834
